# Ablation Study — PIDL Type, Grid Size, Lambda

Systematic ablation on **3 clients, 5 rounds, 2 local epochs** — all 3 datasets.
Results saved to `results_ablation/`. No plots — tabular outputs only.

| Group | What varies | Variants |
|---|---|---|
| 1 — PIDL type | Regularizer | No PIDL · Global PIDL · **Grid-wise 4×4** (baseline) |
| 2 — Grid size | Grid patches | 2×2 · **4×4** (baseline) · 8×8  (λ=0.1 fixed) |
| 3 — Lambda | PIDL weight | λ=0.01 · **λ=0.10** (baseline) · λ=0.50  (grid 4×4 fixed) |

**Baseline** (`gridwise_4x4_lam0.10`) = the 3-client run already in `results/{dataset}/3_clients/`.
It is **loaded directly** — not re-run — so it appears consistently across all three groups.

**New experiments: 6 variants × 3 datasets = 18 runs.**

---
## § 1 — Setup

In [ ]:
# ── Edit your GitHub repo URL here ───────────────────────────────────
GITHUB_REPO = 'https://github.com/PulockDas/medical_fl_pidl.git'
# ──────────────────────────────────────────────────────────────────────

import gc, json, os, sys, time
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('/content/medical_fl_pidl')
if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
    if not PROJECT_ROOT.exists():
        raise RuntimeError('git clone failed — check GITHUB_REPO.')
else:
    os.system(f'git -C {PROJECT_ROOT} pull')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt
!python -c "import flwr, torch, kagglehub; print(f'flwr={flwr.__version__}  torch={torch.__version__}  kagglehub OK')"

import torch
RESULTS_ROOT     = PROJECT_ROOT / 'results_ablation'
MAIN_RESULTS_ROOT = PROJECT_ROOT / 'results'   # existing notebook-1 results
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Ablation results : {RESULTS_ROOT}')
print(f'Main results     : {MAIN_RESULTS_ROOT}')


---
## § 2 — Download Datasets

In [ ]:
import kagglehub
from data.kaggle_loader import find_image_root

DATASET_SLUGS = {
    'brain_tumor_mri':           'masoudnickparvar/brain-tumor-mri-dataset',
    'colon_cancer_or_pathology':  'andrewmvd/lung-and-colon-cancer-histopathological-images',
    'covid':                     'tawsifurrahman/covid19-radiography-database',
}

# ── Colon cancer: choose colon or lung subset ──────────────────────────
COLON_OR_LUNG = 'colon_image_sets'   # or 'lung_image_sets'

DATA_ROOTS = {}

for ds_name, slug in DATASET_SLUGS.items():
    print(f'\nDownloading {ds_name}...')
    raw_path = kagglehub.dataset_download(slug)
    print(f'  Raw path: {raw_path}')

    if ds_name == 'colon_cancer_or_pathology':
        candidates = list(Path(raw_path).rglob(COLON_OR_LUNG))
        if candidates:
            root = str(candidates[0])
        else:
            root = str(find_image_root(raw_path))
    else:
        root = str(find_image_root(raw_path))

    DATA_ROOTS[ds_name] = root
    print(f'  Image root: {root}')

print('\nAll DATA_ROOTS:')
for k, v in DATA_ROOTS.items():
    print(f'  {k}: {v}')


---
## § 3 — Shared Experiment Parameters

In [ ]:
# Fixed across ALL ablation experiments — must match the original notebook-1 run
NUM_CLIENTS      = 3
NUM_ROUNDS       = 5
LOCAL_EPOCHS     = 2
BATCH_SIZE       = 32
LEARNING_RATE    = 0.001
IMAGE_SIZE       = 224
FEATURE_LAYER    = 'layer2'
K                = 1.0
RANDOM_SEED      = 42

# NEW variants only — baseline (gridwise_4x4_lam0.10) is loaded from existing results,
# not re-run, so it stays consistent with the main experiment.
# (label, regularizer_type, lambda_pm, use_grid_loss, grid_size)
VARIANTS = [
    # Group 1: PIDL regularizer type
    ('no_pidl',              'none',         0.00, False, 4),
    ('global_pidl',          'perona_malik', 0.10, False, 4),
    # Group 2: grid size  (lambda fixed at 0.1)
    ('gridwise_2x2_lam0.10', 'perona_malik', 0.10, True,  2),
    ('gridwise_8x8_lam0.10', 'perona_malik', 0.10, True,  8),
    # Group 3: lambda     (grid 4x4 fixed)
    ('gridwise_4x4_lam0.01', 'perona_malik', 0.01, True,  4),
    ('gridwise_4x4_lam0.50', 'perona_malik', 0.50, True,  4),
]

print(f'New variants to run : {len(VARIANTS)} x {len(DATA_ROOTS)} datasets = {len(VARIANTS)*len(DATA_ROOTS)} runs')
print(f'Baseline loaded from: results/{{dataset}}/3_clients/')
print()
for label, rtype, lam, grid, gs in VARIANTS:
    print(f'  {label:<28} reg={rtype:<14} lambda={lam:.2f}  grid={grid}  gs={gs}')


---
## § 4 — Experiment Runner + Baseline Loader

In [ ]:
from flwr.simulation import run_simulation
from federated.server_app import app as _server_app
from federated.client_app import _make_client_app


def _enrich_summary_from_jsonl(log_dir: Path, num_rounds: int, summary: dict) -> dict:
    """Pull ce_loss, reg_loss, training_time, secagg from round_metrics.jsonl."""
    jsonl_path = log_dir / 'round_metrics.jsonl'
    if not jsonl_path.exists():
        return summary
    lines = jsonl_path.read_text().strip().splitlines()
    recs  = [json.loads(l) for l in lines[-num_rounds:] if l.strip()]
    if not recs:
        return summary
    last = recs[-1]
    summary.update({
        'train_ce_loss_final':      round(last.get('train_ce_loss',   0.0), 6),
        'train_reg_loss_final':     round(last.get('train_pidl_loss', 0.0), 6),
        'train_accuracy_final':     round(last.get('train_accuracy',  0.0), 6),
        'training_time_per_round':  round(last.get('elapsed_seconds', 0.0), 2),
        'training_time_total':      round(sum(r.get('elapsed_seconds', 0) for r in recs), 2),
        'secagg_overhead_mean_sec': round(
            sum(r.get('secagg_overhead_sec', 0) for r in recs) / max(len(recs), 1), 4),
    })
    return summary


def _build_summary_dict(log_dir: Path, dataset_name: str, variant_label: str) -> dict:
    """Construct a full metrics summary from fl_rounds.csv + round_metrics.jsonl."""
    csv_path = log_dir / 'fl_rounds.csv'
    summary  = {'dataset': dataset_name, 'variant': variant_label}

    if not csv_path.exists():
        print(f'  [WARN] fl_rounds.csv missing: {csv_path}')
        return summary

    df = pd.read_csv(csv_path)
    df = df[df['round'] > 0].reset_index(drop=True)
    if df.empty:
        return summary

    last         = df.iloc[-1]
    best_acc_idx = df['global_test_acc'].idxmax()
    best         = df.iloc[best_acc_idx]

    def _f(v): return round(float(v), 6)

    summary.update({
        'num_rounds':              int(df['round'].max()),
        'final_accuracy':          _f(last['global_test_acc']),
        'best_accuracy':           _f(best['global_test_acc']),
        'best_accuracy_round':     int(best['round']),
        'final_balanced_accuracy': _f(last['balanced_accuracy']),
        'best_balanced_accuracy':  _f(df['balanced_accuracy'].max()),
        'final_macro_f1':          _f(last['f1_macro']),
        'best_macro_f1':           _f(df['f1_macro'].max()),
        'final_micro_f1':          _f(last['f1_micro']),
        'final_weighted_f1':       _f(last['f1_weighted']),
        'final_precision_macro':   _f(last['precision_macro']),
        'final_recall_macro':      _f(last['recall_macro']),
        'final_specificity_macro': _f(last['specificity_macro']),
        'final_sensitivity_macro': _f(last['sensitivity_macro']),
        'final_roc_auc_macro':     _f(last['roc_auc_macro']),
        'best_roc_auc_macro':      _f(df['roc_auc_macro'].max()),
        'final_pr_auc_macro':      _f(last['pr_auc_macro']),
        'final_ece':               _f(last['ece']),
        'final_mean_confidence':   _f(last['mean_confidence']),
        'final_mean_entropy':      _f(last['mean_entropy']),
        'final_test_loss':         _f(last['global_test_loss']),
        'inference_time_total_sec':_f(df['inference_time_sec'].sum()),
        'inference_time_per_round':_f(df['inference_time_sec'].mean()),
    })

    summary = _enrich_summary_from_jsonl(log_dir, NUM_ROUNDS, summary)
    return summary


def load_baseline_results() -> list:
    """Load gridwise_4x4_lam0.10 results from the existing 3-client main experiment."""
    baselines = []
    variant_label = 'gridwise_4x4_lam0.10'
    for ds_name in DATA_ROOTS:
        src_dir = MAIN_RESULTS_ROOT / ds_name / '3_clients'
        if not src_dir.exists():
            print(f'  [WARN] Baseline not found: {src_dir}  — skipping {ds_name}')
            continue
        summary = _build_summary_dict(src_dir, ds_name, variant_label)
        # Mirror files to results_ablation so everything is in one place
        dst_dir = RESULTS_ROOT / ds_name / variant_label
        dst_dir.mkdir(parents=True, exist_ok=True)
        for fname in ['fl_rounds.csv', 'round_metrics.jsonl', 'per_class_metrics.csv',
                      'config.json', 'dataset_summary.json']:
            src_f = src_dir / fname
            if src_f.exists():
                import shutil
                shutil.copy2(src_f, dst_dir / fname)
        (dst_dir / 'ablation_summary.json').write_text(json.dumps(summary, indent=2))
        print(f'  [BASE] {ds_name}/gridwise_4x4_lam0.10 loaded  '
              f'(acc={summary.get("final_accuracy","?"):.4f})')
        baselines.append(summary)
    return baselines


def run_ablation_experiment(
    dataset_name: str,
    variant_label: str,
    regularizer_type: str,
    lambda_pm: float,
    use_grid_loss: bool,
    grid_size: int,
    resume: bool = True,
) -> dict:
    """Run one ablation experiment and return a metrics summary dict."""
    log_dir = RESULTS_ROOT / dataset_name / variant_label
    log_dir.mkdir(parents=True, exist_ok=True)

    # Resume: skip if fl_rounds.csv already has NUM_ROUNDS rows beyond round 0
    csv_path = log_dir / 'fl_rounds.csv'
    if resume and csv_path.exists():
        try:
            df_check = pd.read_csv(csv_path)
            if len(df_check[df_check['round'] > 0]) >= NUM_ROUNDS:
                print(f'  [SKIP] {dataset_name}/{variant_label} — already done.')
                return _build_summary_dict(log_dir, dataset_name, variant_label)
        except Exception:
            pass

    run_cfg = {
        'dataset_name':      dataset_name,
        'data_root':         DATA_ROOTS[dataset_name],
        'log_dir':           str(log_dir),
        'num_classes':       0,
        'num_clients':       NUM_CLIENTS,
        'num_server_rounds': NUM_ROUNDS,
        'min_fit_clients':   NUM_CLIENTS,
        'local_epochs':      LOCAL_EPOCHS,
        'batch_size':        BATCH_SIZE,
        'learning_rate':     LEARNING_RATE,
        'image_size':        IMAGE_SIZE,
        'feature_layer':     FEATURE_LAYER,
        'regularizer_type':  regularizer_type,
        'lambda_pm':         lambda_pm,
        'use_grid_loss':     use_grid_loss,
        'grid_size':         grid_size,
        'k':                 K,
        'random_seed':       RANDOM_SEED,
        'use_secagg':        False,
        'enable_attack':     False,
        'enable_update_clipping': False,
    }

    os.environ['FL_RUN_OVERRIDE'] = json.dumps(run_cfg)
    _client_app = _make_client_app()

    gpu_frac    = 0.5 if torch.cuda.is_available() else 0.0
    backend_cfg = {'client_resources': {'num_cpus': 2, 'num_gpus': gpu_frac}}

    print(f'  [RUN ] {dataset_name}/{variant_label}')
    t0      = time.time()
    success = False
    try:
        run_simulation(
            server_app    =_server_app,
            client_app    =_client_app,
            num_supernodes=NUM_CLIENTS,
            backend_config=backend_cfg,
        )
        elapsed = time.time() - t0
        print(f'  [OK  ] {elapsed:.0f}s')
        try:
            from federated.server_app import finalize_experiment
            finalize_experiment()
        except Exception as fe:
            print(f'  [WARN] finalize_experiment: {fe}')
        success = True
    except Exception as exc:
        elapsed = time.time() - t0
        print(f'  [FAIL] {elapsed:.0f}s — {exc}')
        import traceback; traceback.print_exc()
    finally:
        os.environ.pop('FL_RUN_OVERRIDE', None)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if not success:
        return {'dataset': dataset_name, 'variant': variant_label, 'error': True}

    summary = _build_summary_dict(log_dir, dataset_name, variant_label)
    (log_dir / 'ablation_summary.json').write_text(json.dumps(summary, indent=2))
    return summary


print('Runner and baseline loader defined.')


---
## § 5 — Load Baseline (Existing 3-Client Results)

The `gridwise_4x4_lam0.10` variant is loaded directly from `results/{dataset}/3_clients/`.
Files are mirrored into `results_ablation/` so the full ablation folder is self-contained.


In [ ]:
all_results = load_baseline_results()
print(f'\nBaselines loaded: {len(all_results)}')


---
## § 6 — Run New Ablation Experiments

18 new experiments (6 variants × 3 datasets).  
Already-completed runs are skipped automatically.


In [ ]:
total   = len(VARIANTS) * len(DATA_ROOTS)
run_idx = 0

for label, rtype, lam, use_grid, gs in VARIANTS:
    print(f'\n=== Variant: {label} ===')
    for ds_name in DATA_ROOTS:
        run_idx += 1
        print(f'--- Run {run_idx}/{total} | {ds_name} | {label} ---')
        result = run_ablation_experiment(
            dataset_name     = ds_name,
            variant_label    = label,
            regularizer_type = rtype,
            lambda_pm        = lam,
            use_grid_loss    = use_grid,
            grid_size        = gs,
            resume           = True,
        )
        all_results.append(result)

print(f'\nTotal results collected: {len(all_results)} (3 baseline + {run_idx} new)')


---
## § 7 — Generate Summary Tables

In [ ]:
master_df = pd.DataFrame([r for r in all_results if 'error' not in r])

if master_df.empty:
    print('No results to summarise yet.')
else:
    GROUP_MAP = {
        'no_pidl':               'group1_pidl_type',
        'global_pidl':           'group1_pidl_type',
        'gridwise_4x4_lam0.10':  'baseline',          # appears in all groups
        'gridwise_2x2_lam0.10':  'group2_grid_size',
        'gridwise_8x8_lam0.10':  'group2_grid_size',
        'gridwise_4x4_lam0.01':  'group3_lambda',
        'gridwise_4x4_lam0.50':  'group3_lambda',
    }
    master_df['group'] = master_df['variant'].map(GROUP_MAP)

    master_path = RESULTS_ROOT / 'ablation_summary.csv'
    master_df.to_csv(master_path, index=False)
    print(f'Saved: ablation_summary.csv  ({len(master_df)} rows)')

    KEY_COLS = [
        'dataset', 'variant',
        'final_accuracy', 'best_accuracy', 'final_balanced_accuracy',
        'final_macro_f1', 'best_macro_f1', 'final_weighted_f1',
        'final_precision_macro', 'final_recall_macro',
        'final_specificity_macro', 'final_sensitivity_macro',
        'final_roc_auc_macro', 'final_pr_auc_macro',
        'final_ece', 'final_mean_confidence', 'final_mean_entropy',
        'final_test_loss',
        'train_ce_loss_final', 'train_reg_loss_final', 'train_accuracy_final',
        'training_time_total', 'training_time_per_round',
        'inference_time_total_sec', 'inference_time_per_round',
        'secagg_overhead_mean_sec',
    ]
    available = [c for c in KEY_COLS if c in master_df.columns]

    # ── Group 1: PIDL type ─────────────────────────────────────────────
    VARIANT_ORDER_G1 = ['no_pidl', 'global_pidl', 'gridwise_4x4_lam0.10']
    g1 = master_df[master_df['variant'].isin(VARIANT_ORDER_G1)][available].copy()
    g1['variant'] = pd.Categorical(g1['variant'], VARIANT_ORDER_G1, ordered=True)
    g1.sort_values(['dataset', 'variant']).to_csv(
        RESULTS_ROOT / 'group1_pidl_type.csv', index=False)
    print('Saved: group1_pidl_type.csv')

    # ── Group 2: Grid size ─────────────────────────────────────────────
    VARIANT_ORDER_G2 = ['gridwise_2x2_lam0.10', 'gridwise_4x4_lam0.10', 'gridwise_8x8_lam0.10']
    g2 = master_df[master_df['variant'].isin(VARIANT_ORDER_G2)][available].copy()
    g2['variant'] = pd.Categorical(g2['variant'], VARIANT_ORDER_G2, ordered=True)
    g2.sort_values(['dataset', 'variant']).to_csv(
        RESULTS_ROOT / 'group2_grid_size.csv', index=False)
    print('Saved: group2_grid_size.csv')

    # ── Group 3: Lambda ────────────────────────────────────────────────
    VARIANT_ORDER_G3 = ['gridwise_4x4_lam0.01', 'gridwise_4x4_lam0.10', 'gridwise_4x4_lam0.50']
    g3 = master_df[master_df['variant'].isin(VARIANT_ORDER_G3)][available].copy()
    g3['variant'] = pd.Categorical(g3['variant'], VARIANT_ORDER_G3, ordered=True)
    g3.sort_values(['dataset', 'variant']).to_csv(
        RESULTS_ROOT / 'group3_lambda.csv', index=False)
    print('Saved: group3_lambda.csv')

    print()
    print('=== Final accuracy (all variants) ===')
    if 'final_accuracy' in master_df.columns:
        piv = master_df.pivot_table(
            index='variant', columns='dataset',
            values='final_accuracy', aggfunc='first'
        )
        print(piv.to_string())


---
## § 8 — Download Results

In [ ]:
import subprocess
subprocess.run(['zip', '-r', '/content/ablation_results.zip', str(RESULTS_ROOT)], check=True)

from google.colab import files
files.download('/content/ablation_results.zip')
print('Download started.')
